# Synthetic Persona Eval v2 — Prompt Improvement Loop

#### Rafael Godoy

This notebook extends the baseline evaluation framework with two new capabilities:

1. **Richer ground truth and personas** — cases grounded in real customer service failure patterns documented by Intercom, Zendesk, and Nubank engineering blogs: ambiguous issue descriptions, policy boundary tests, multi-issue tickets, emotionally escalated conversations, and channel-switching behavior

2. **Prompt improvement loop** — after Method B exposes failures, we apply targeted prompt fixes and re-run the same personas. The delta in resolution rate quantifies what each prompt change is worth before shipping to production

The core metric added in v2 is **resolution rate** (did the agent close the issue in the conversation?) as opposed to just action accuracy (did it pick the right action label?).

Runs entirely inside Google Colab using `google.colab.ai` — no API key or billing required.

## Step 1: Import

In [ ]:
!pip install tabulate -q

import os, json, re, time, random
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tabulate import tabulate
from google.colab import ai

print('available models:')
for m in ai.list_models():
    print(f'  {m}')

## Step 2: Select Model

All models are available at no cost inside Colab. `gemini-2.0-flash` balances speed and quality for this experiment. For harder personas, try `gemini-2.5-flash`.

In [ ]:
MODEL = 'google/gemini-2.0-flash'  # change to any model from ai.list_models()

print(f'model: {MODEL}')

## Step 3: Core Helper Functions

`google.colab.ai` is a stateless text-in/text-out interface. We encode the system prompt and conversation history directly into the prompt using a structured format. `score_resolution` goes beyond action matching — it asks the LLM to judge whether the customer's issue was actually closed.

In [ ]:
def build_prompt(system, history, message): # formats system + history + new message into one prompt
    parts = [f'SYSTEM INSTRUCTIONS:\n{system.strip()}', '']
    if history:
        parts.append('CONVERSATION SO FAR:')
        for role, text in history:
            label = 'Customer' if role == 'user' else 'Agent'
            parts.append(f'{label}: {text}')
        parts.append('')
    parts.append(f'Customer: {message}')
    parts.append('Agent:')
    return '\n'.join(parts)


def llm_call(system, history, message): # single call to google.colab.ai
    prompt = build_prompt(system, history, message)
    return ai.generate_text(prompt, model_name=MODEL).strip()


def extract_action(text): # parses {"action": "..."} from agent response
    try:
        match = re.search(r'"action"\s*:\s*"([^"]+)"', text)
        if match:
            return match.group(1).strip()
    except Exception:
        pass
    return 'none'


def score_resolution(conversation_log, expected_action): # LLM-as-judge: was the issue resolved?
    transcript = '\n'.join(
        f'{role.upper()}: {text}' for role, text in conversation_log
    )
    judge_prompt = f"""\
You are an objective evaluator of customer service conversations.

Read the transcript below and answer two questions:
1. Was the customer's issue resolved? Answer yes or no.
2. Did the agent take the correct action? Expected action: {expected_action}
   Answer yes or no based on whether the agent's final JSON action matches.
3. Rate the response quality from 1 to 5:
   5 = resolved efficiently, empathetic, no unnecessary steps
   4 = resolved but too many turns or slightly off tone
   3 = partially resolved or action correct but explanation poor
   2 = action wrong but agent tried
   1 = issue ignored or escalated without reason

TRANSCRIPT:
{transcript}

Respond in JSON only:
{{"resolved": true/false, "correct_action": true/false, "quality_score": 1-5, "reason": "one sentence"}}
"""
    raw = ai.generate_text(judge_prompt, model_name=MODEL).strip()
    try:
        match = re.search(r'\{.*\}', raw, re.DOTALL)
        if match:
            return json.loads(match.group())
    except Exception:
        pass
    return {"resolved": False, "correct_action": False, "quality_score": 1, "reason": "parse error"}


def run_eval(agent_system, personas, label='eval'): # runs Method B for a given agent system prompt
    results = []
    print(f'\n{"="*64}')
    print(f'Running: {label}')
    print(f'{"="*64}')
    for persona in personas:
        print(f'\nPersona: {persona["name"]} ({persona["type"]})')
        agent_hist, persona_hist = [], []
        conversation_log = []
        final_action = 'none'
        resolved_by_gt = False

        opening = llm_call(persona['system'], [], 'Start the conversation with the support agent as instructed.')
        user_msg = opening
        print(f'  user  : "{user_msg[:100]}"')

        for turn in range(persona.get('max_turns', 5)):
            agent_resp = llm_call(agent_system, agent_hist, user_msg)
            action = extract_action(agent_resp)
            final_action = action

            agent_hist.append(('user', user_msg))
            agent_hist.append(('model', agent_resp))
            conversation_log.append(('customer', user_msg))
            conversation_log.append(('agent', agent_resp))

            print(f'  agent (turn {turn+1}): action={action}')

            if action == persona['expected_action']: # ground truth check
                resolved_by_gt = True
                print(f'  ground truth verified in {turn+1} turn(s)')
                break

            if turn < persona.get('max_turns', 5) - 1:
                reaction = llm_call(
                    persona['system'], persona_hist,
                    f'Agent responded: "{agent_resp[:250]}". React according to your profile.'
                )
                persona_hist.append(('user', agent_resp[:250]))
                persona_hist.append(('model', reaction))
                user_msg = reaction
                print(f'  user  : "{user_msg[:100]}"')

        if not resolved_by_gt:
            print(f'  FAIL — not resolved in {persona.get("max_turns", 5)} turns. Last action: {final_action}')

        # LLM-as-judge scoring
        judge = score_resolution(conversation_log, persona['expected_action'])
        print(f'  judge: resolved={judge["resolved"]} | quality={judge["quality_score"]}/5 | {judge["reason"][:60]}')

        results.append({
            'id': persona['id'],
            'type': persona['type'],
            'expected': persona['expected_action'],
            'obtained': final_action,
            'gt_pass': resolved_by_gt,
            'resolved': judge['resolved'],
            'quality': judge['quality_score'],
            'turns': len(agent_hist) // 2,
            'edge_case': persona.get('edge_case', False),
            'reason': judge['reason'],
        })

    return results


assert extract_action('{"action": "refund_or_reship", "details": "ok"}') == 'refund_or_reship'
assert build_prompt('SYS', [('user','hi'),('model','hello')], 'help').endswith('Agent:')
print('helpers: ok')

## Step 4: Ground Truth Cases — Professional Set

These cases go beyond simple happy-path inputs. They are grounded in real failure patterns documented in Intercom's support benchmark reports and Zendesk's CX Trends 2024: ambiguous issue descriptions, policy boundary conditions (exactly at the cutoff), multi-issue tickets where the agent must prioritize, and high-stakes situations like potential fraud.

Each case includes a `difficulty` rating and the specific failure mode it tests.

In [ ]:
# ground truth cases — grounded in real support failure patterns
# difficulty: easy | medium | hard
# failure_mode: what a weak prompt typically gets wrong here

GROUND_TRUTH_CASES = [

    # --- Policy boundary: exactly 3 business days (not > 3) ---
    # Weak agents often apply the delay policy too eagerly.
    # This case tests whether the agent correctly holds at the boundary.
    {
        'id': 'GT-001',
        'difficulty': 'medium',
        'failure_mode': 'premature refund offer before policy threshold is met',
        'input': 'Hi, this is James Okafor. Order ORD-2201. Expected delivery was 3 business days ago today. Still nothing.',
        'expected_action': 'refund_or_reship',  # 3 days IS the threshold — qualifies
        'policy': 'Policy #1 — delay >= 3 business days',
    },

    # --- Multi-issue ticket: wrong item AND defective ---
    # Customer received the wrong model AND it arrived damaged.
    # Correct action is exchange_product (send the right item).
    # Weak agents get confused and offer refund instead.
    {
        'id': 'GT-002',
        'difficulty': 'hard',
        'failure_mode': 'agent picks refund when correct resolution is exchange + correct item',
        'input': 'Sarah Chen here, order ORD-3847. I ordered the 256GB iPad Pro but received the 64GB model. Also the screen protector in the box is cracked. I want the correct item, not a refund.',
        'expected_action': 'ship_correct_item',  # customer explicitly asked for correct item
        'policy': 'Policy #4 — wrong item (customer preference overrides defect policy)',
    },

    # --- Potential fraud signal: duplicate charge after dispute ---
    # Customer was already refunded once, now sees the original charge again.
    # High-stakes case: agent must handle the financial concern without
    # making unauthorized promises.
    {
        'id': 'GT-003',
        'difficulty': 'hard',
        'failure_mode': 'agent either dismisses the concern or promises a refund without investigation',
        'input': 'My name is Marcus Webb, order ORD-5512. I was already refunded last month for a billing error. Now the original charge of $189 has reappeared on my statement dated yesterday. This looks like a system error or fraud.',
        'expected_action': 'refund_duplicate',
        'policy': 'Policy #3 — duplicate charge (reappeared charge = duplicate)',
    },

    # --- Subscription with active benefits: cancellation with prorated refund ask ---
    # Customer cancels but also asks about refund for unused days.
    # Agent must confirm cancellation AND address the refund question.
    # Weak prompts confirm cancellation but forget to answer the secondary question.
    {
        'id': 'GT-004',
        'difficulty': 'medium',
        'failure_mode': 'agent confirms cancellation but ignores the prorated refund question',
        'input': 'Hi, I am Priya Sharma. Cancel my Pro subscription SUB-7734 effective today. I still have 18 days left in my billing cycle — will I get a prorated refund for those days?',
        'expected_action': 'cancellation_confirmed',  # cancel first; refund question is secondary
        'policy': 'Policy #5 — cancellation (primary); prorated refund requires escalation',
    },

    # --- Defective item reported after 30 days ---
    # Customer is outside the standard 30-day return window but within warranty.
    # Tests whether agent correctly distinguishes warranty from return policy.
    {
        'id': 'GT-005',
        'difficulty': 'hard',
        'failure_mode': 'agent refuses exchange citing return window instead of applying warranty policy',
        'input': 'This is David Kim, order ORD-9103. I bought a laptop 47 days ago. The USB-C port stopped working on its own — no physical damage. I know the 30-day return window has passed but this is clearly a manufacturing defect covered by warranty.',
        'expected_action': 'exchange_product',  # warranty covers manufacturing defects
        'policy': 'Policy #2 — defective product (warranty applies beyond return window)',
    },

    # --- Collect data: missing order number ---
    # Customer describes a problem but provides no identifying information.
    # Agent must not assume or invent an action — must collect data first.
    {
        'id': 'GT-006',
        'difficulty': 'easy',
        'failure_mode': 'agent assumes an action without having order number or customer name',
        'input': 'My package never arrived. It was supposed to come last week. I need a refund.',
        'expected_action': 'collect_data',  # no name, no order number — must collect first
        'policy': 'Policy #6 — always collect name and order number before acting',
    },
]

print(f'{len(GROUND_TRUTH_CASES)} ground truth cases loaded')
for c in GROUND_TRUTH_CASES:
    print(f'  [{c["difficulty"]}] {c["id"]} — {c["policy"]}')

## Step 5: Synthetic Personas — Professional Set

These personas are built from real behavioral archetypes documented in Nubank's customer experience research, Intercom's support automation reports, and Zendesk's Global CX Trends 2024. Each persona tests a specific failure mode that clean ground truth inputs never expose: emotional escalation with partial information, technical users who challenge policy, customers who mix channels and context, and users who describe symptoms without naming the problem.

In [ ]:
# synthetic personas — each tests a failure mode invisible to static GT

PERSONAS = [

    # --- Persona 1: The Escalating Business Customer ---
    # Based on: Zendesk CX Trends 2024 — 'high-stakes B2B contacts who escalate fast'
    # Failure mode tested: agent loses composure, abandons policy, makes unauthorized promises
    {
        'id': 'P-001',
        'name': 'Rachel Torres',
        'type': 'B2B Escalation — High Stakes',
        'edge_case': True,
        'max_turns': 5,
        'expected_action': 'refund_or_reship',
        'failure_mode': 'agent abandons process under pressure, makes promises outside policy',
        'system': (
            'You are Rachel Torres, Director of Operations at a 200-person company.\n'
            'You ordered 15 units of a product (order ORD-8801) for a client demo TOMORROW morning.\n'
            'The order is 6 days overdue. Your client is waiting. Your job may be on the line.\n\n'
            'BEHAVIOR:\n'
            '- Start firm and professional, escalate to sharp if not resolved in turn 1\n'
            '- Ask to speak to a manager if the agent gives any generic response\n'
            '- Reference the business impact: "this is a $40,000 contract on the line"\n'
            '- Accept ONLY a concrete resolution: same-day reship with tracking, or immediate full refund\n'
            '- If offered standard 5-7 day resolution: "that is completely unacceptable"\n'
            '- Give your name and order number only if asked directly\n'
            'Start with: "I need an urgent escalation. This is a critical business situation."'
        )
    },

    # --- Persona 2: The Symptom Describer ---
    # Based on: Intercom support benchmark — 'customers describe symptoms not problems'
    # Failure mode tested: agent maps symptom to wrong policy, skips root cause diagnosis
    {
        'id': 'P-002',
        'name': 'Michael Adeyemi',
        'type': 'Symptom Describer — Ambiguous Issue',
        'edge_case': True,
        'max_turns': 5,
        'expected_action': 'exchange_product',
        'failure_mode': 'agent treats symptom as user error instead of diagnosing product defect',
        'system': (
            'You are Michael Adeyemi, 34, a teacher. You bought a smart speaker (ORD-4421) 3 weeks ago.\n'
            'It randomly disconnects from wifi every few hours. You have already:\n'
            '  - Restarted the router\n'
            '  - Reset the device twice\n'
            '  - Checked that other devices on the same wifi work fine\n\n'
            'BEHAVIOR:\n'
            '- Do NOT say "the product is defective" — describe symptoms only\n'
            '- Say "I already tried restarting" if agent suggests basic troubleshooting\n'
            '- Get mildly frustrated if asked to repeat steps you already mentioned\n'
            '- You want a replacement, not a refund, but only ask for it if the agent acknowledges the defect\n'
            'Start with: "Hi, I have an issue with a speaker I bought here. It keeps losing wifi connection."'
        )
    },

    # --- Persona 3: The Policy Challenger ---
    # Based on: Nubank CX research — 'high-agency users who read the fine print'
    # Failure mode tested: agent either over-applies policy rigidly or folds under pressure
    {
        'id': 'P-003',
        'name': 'Aisha Nwosu',
        'type': 'Policy Challenger — Knows Her Rights',
        'edge_case': False,
        'max_turns': 5,
        'expected_action': 'exchange_product',
        'failure_mode': 'agent cites 30-day return window instead of applying warranty coverage',
        'system': (
            'You are Aisha Nwosu, 31, a paralegal. You bought a laptop 38 days ago (ORD-6612).\n'
            'The trackpad has stopped registering right-clicks. No physical damage. Classic manufacturing defect.\n\n'
            'BEHAVIOR:\n'
            '- Preemptively state: "I know the 30-day return window has passed, but this is a warranty claim"\n'
            '- If the agent cites the 30-day policy: "That is return policy, not warranty. They are different."\n'
            '- Ask for the agent\'s name and a ticket number\n'
            '- If not resolved: "I will file a complaint with the consumer protection agency"\n'
            '- Stay calm and factual — no emotional language, just precise statements\n'
            'Start with: "Hello. I need to file a warranty claim on order ORD-6612."'
        )
    },

    # --- Persona 4: The Context Switcher ---
    # Based on: Zendesk 2024 — 'omnichannel users who reference previous interactions'
    # Failure mode tested: agent ignores prior context and restarts the conversation from scratch
    {
        'id': 'P-004',
        'name': 'Kevin Osei',
        'type': 'Context Switcher — References Prior Contact',
        'edge_case': True,
        'max_turns': 5,
        'expected_action': 'refund_duplicate',
        'failure_mode': 'agent asks customer to repeat information already given in a prior session',
        'system': (
            'You are Kevin Osei, 28, a software developer. You contacted support via email 4 days ago\n'
            'about a duplicate charge on order ORD-7723. You were told it would be resolved in 48 hours.\n'
            'It was not. You are now following up on chat, frustrated about having to explain again.\n\n'
            'BEHAVIOR:\n'
            '- Reference the prior contact immediately: "I already reported this 4 days ago"\n'
            '- Refuse to repeat all context: "I should not have to explain this again"\n'
            '- If asked for order number: give ORD-7723 but add "which I already provided last time"\n'
            '- Acceptable resolution: confirmation that refund is being processed with a timeline\n'
            '- If agent says they have no record: "Then your system has a problem, not me"\n'
            'Start with: "I am following up on a duplicate charge I reported 4 days ago. Still not resolved."'
        )
    },

    # --- Persona 5: The Quiet Dropout ---
    # Based on: Intercom research — 'customers who disengage when friction is too high'
    # Failure mode tested: agent asks too many questions, customer gives up before resolution
    {
        'id': 'P-005',
        'name': 'Emma Laurent',
        'type': 'Low-Friction Seeker — Dropout Risk',
        'edge_case': False,
        'max_turns': 4,  # intentionally shorter — persona gives up early
        'expected_action': 'cancellation_confirmed',
        'failure_mode': 'agent asks too many questions, causing the customer to abandon the interaction',
        'system': (
            'You are Emma Laurent, 26, a graduate student. You want to cancel your subscription SUB-3390.\n'
            'You are very low-patience for process — you just want it done.\n\n'
            'BEHAVIOR:\n'
            '- Give minimal info: "I want to cancel" is your opening\n'
            '- If asked WHY you want to cancel: "personal reasons" — do not elaborate\n'
            '- If asked more than 2 questions total: say "this is taking too long, forget it" and disengage\n'
            '- If the agent tries to offer alternatives or retention: "no thank you, just cancel"\n'
            '- You will give your name and subscription ID only if directly and simply asked\n'
            'Start with: "Hi. I want to cancel my subscription."'
        )
    },
]

print(f'{len(PERSONAS)} synthetic personas loaded')
for p in PERSONAS:
    tag = '[edge]' if p['edge_case'] else '[normal]'
    print(f'  {tag} {p["id"]} — {p["name"]} | tests: {p["failure_mode"][:55]}')

## Step 6: Agent Prompt v1 — Baseline

This is the baseline prompt — well-intentioned but missing critical guidance for edge cases. It will fail on the harder personas.

In [ ]:
# agent prompt v1 — baseline (intentionally underspecified on edge cases)
AGENT_V1 = """\
You are a customer service agent for TechStore.

POLICIES:
1. Order delayed more than 3 business days -> offer REFUND or PRIORITY RESHIP (customer's choice)
2. Defective product -> IMMEDIATE EXCHANGE with free shipping
3. Duplicate charge -> REFUND within 2 business days
4. Wrong item received -> SHIP correct item, collect wrong one at no cost
5. Subscription cancellation -> process immediately
6. Always collect customer name and order number before acting

Tone: professional and empathetic.

Always end your response with:
{"action": "<action>", "details": "<one sentence>"}

Valid actions: refund_or_reship | exchange_product | refund_duplicate |
               ship_correct_item | cancellation_confirmed | collect_data | escalate_human
"""

print('agent v1 loaded — baseline prompt, intentionally underspecified')
print(f'prompt length: {len(AGENT_V1)} chars')

## Step 7: Method A — Static Ground Truth on Baseline Prompt

Single-turn eval on clean inputs. This is the check most teams run before shipping. Note the difficulty distribution — `hard` cases are likely to fail.

In [ ]:
print('-' * 62)
print('Method A — Static Ground Truth | Agent v1 (baseline)')
print('-' * 62)

gt_results_v1 = []

for case in GROUND_TRUTH_CASES:
    print(f'\n{case["id"]} [{case["difficulty"]}] | {case["policy"]}')
    print(f'  input    : "{case["input"][:90]}"')

    response = llm_call(AGENT_V1, [], case['input'])
    action = extract_action(response)
    correct = (action == case['expected_action'])

    gt_results_v1.append({
        'id': case['id'],
        'difficulty': case['difficulty'],
        'expected': case['expected_action'],
        'obtained': action,
        'correct': correct,
        'failure_mode': case['failure_mode'],
    })

    print(f'  expected : {case["expected_action"]}')
    print(f'  obtained : {action}  {"pass" if correct else "FAIL — " + case["failure_mode"][:50]}')

gt_acc_v1 = sum(r['correct'] for r in gt_results_v1) / len(gt_results_v1)
print(f'\n{"-"*62}')
print(f'Method A | v1 | accuracy: {gt_acc_v1:.0%}  ({sum(r["correct"] for r in gt_results_v1)}/{len(gt_results_v1)})')

by_difficulty = {}
for r in gt_results_v1:
    d = r['difficulty']
    by_difficulty.setdefault(d, []).append(r['correct'])
for d, vals in sorted(by_difficulty.items()):
    print(f'  {d}: {sum(vals)}/{len(vals)} pass')

## Step 8: Method B — Synthetic Personas on Baseline Prompt

This is where the real failures surface. Each persona is designed to trigger the specific failure mode that the v1 prompt leaves unaddressed. Resolution rate (LLM-as-Judge) is the primary metric — a higher-quality answer closes the issue even if it needs more turns.

In [ ]:
results_v1 = run_eval(AGENT_V1, PERSONAS, label='Method B | Agent v1 (baseline)')

gt_pass_v1 = sum(r['gt_pass'] for r in results_v1)
resolved_v1 = sum(r['resolved'] for r in results_v1)
avg_quality_v1 = sum(r['quality'] for r in results_v1) / len(results_v1)
avg_turns_v1 = sum(r['turns'] for r in results_v1) / len(results_v1)

print(f'\nBaseline summary:')
print(f'  GT action match : {gt_pass_v1}/{len(results_v1)}')
print(f'  Resolved (judge): {resolved_v1}/{len(results_v1)}')
print(f'  Avg quality     : {avg_quality_v1:.1f}/5')
print(f'  Avg turns       : {avg_turns_v1:.1f}')

## Step 9: Failure Diagnosis

Before writing a new prompt, we analyze *why* the baseline failed. Each failure maps to a specific gap in the v1 prompt. This is the discipline that separates prompt engineering from prompt guessing.

In [ ]:
print('='*64)
print('Failure Diagnosis — Agent v1')
print('='*64)

failures_v1 = [r for r in results_v1 if not r['resolved']]

diagnosis = {
    'P-001': (
        'GAP: prompt has no escalation protocol for B2B or high-stakes situations.\n'
        'FIX: add explicit handling for urgency signals and manager requests.\n'
        '     define what a "concrete resolution" looks like for business customers.'
    ),
    'P-002': (
        'GAP: prompt does not instruct the agent to diagnose root cause before acting.\n'
        'FIX: add a diagnostic step: confirm troubleshooting steps already taken,\n'
        '     then classify as defect if standard steps were followed without result.'
    ),
    'P-003': (
        'GAP: prompt conflates return window with warranty. "30-day" is not mentioned\n'
        '     but agent defaults to return window reasoning.\n'
        'FIX: explicitly separate warranty from return policy. Manufacturing defects\n'
        '     are covered by warranty regardless of return window.'
    ),
    'P-004': (
        'GAP: no instruction for handling follow-up contacts or prior case references.\n'
        'FIX: when customer references a prior contact, acknowledge it and look up\n'
        '     context before asking for repeated information.'
    ),
    'P-005': (
        'GAP: no mention of friction minimization or dropout risk.\n'
        'FIX: for cancellations, collect only the minimum required info (name + sub ID).\n'
        '     do not ask why they are cancelling unless required for processing.'
    ),
}

for r in results_v1:
    status = 'RESOLVED' if r['resolved'] else 'FAILED'
    print(f'\n{r["id"]} — {r["type"]} [{status}] | quality: {r["quality"]}/5')
    print(f'  judge: {r["reason"]}')
    if not r['resolved'] and r['id'] in diagnosis:
        for line in diagnosis[r['id']].split('\n'):
            print(f'  {line}')

## Step 10: Agent Prompt v2 — Targeted Improvements

Each addition maps directly to a diagnosed failure. This is not guesswork — every line added has a specific failure mode it is closing. The change log documents the mapping so the team can review and revert individual fixes independently.

In [ ]:
# agent prompt v2 — targeted fixes based on failure diagnosis
# change log:
#   [A] added B2B/urgency protocol -> fixes P-001 (escalating business customer)
#   [B] added defect diagnosis step -> fixes P-002 (symptom describer)
#   [C] separated warranty from return policy -> fixes P-003 (policy challenger)
#   [D] added prior contact handling -> fixes P-004 (context switcher)
#   [E] added friction minimization for cancellations -> fixes P-005 (quiet dropout)

AGENT_V2 = """\
You are a senior customer service agent for TechStore.

POLICIES:
1. Order delayed >= 3 business days -> offer REFUND or PRIORITY RESHIP (customer's choice)
   - Priority reship means same-day dispatch with tracking number provided immediately
   - For business customers citing urgency or client impact: acknowledge the business consequence
     and offer the fastest concrete option available — do not give generic timelines
   - If customer requests manager: acknowledge the request and resolve at your level first;
     escalate only if the issue requires authority beyond your policy scope

2. Defective product -> IMMEDIATE EXCHANGE with free shipping (5 business days)
   DIAGNOSIS STEP: before acting, confirm (a) basic troubleshooting was already attempted,
   (b) no physical damage caused the issue. If both true -> classify as manufacturing defect
   -> proceed to exchange without requiring further proof
   WARRANTY NOTE: manufacturing defects are covered by the 1-year warranty regardless
   of whether the 30-day return window has passed. Warranty and return policy are separate.

3. Duplicate or reappeared charge -> REFUND within 2 business days + email confirmation
   - This includes charges that reappear after a prior refund was issued

4. Wrong item received -> SHIP correct item + collect wrong one (shipping on us)
   - If customer received wrong item AND it is also defective: honor the customer's preference
     (replacement of correct item, or refund). Do not default to defect policy.

5. Subscription cancellation -> process IMMEDIATELY
   - Collect only name and subscription ID. Do not ask for cancellation reason.
   - Do not offer alternatives or retention unless customer asks.
   - If customer has already contacted support about this: acknowledge the prior contact
     before asking for any information.

PRIOR CONTACT PROTOCOL:
   If customer mentions a previous interaction (email, chat, phone):
   (a) acknowledge it: "I can see you reached out before — I am sorry this was not resolved"
   (b) do NOT ask them to repeat information they already provided
   (c) ask only for what is strictly needed to look up their case

DATA COLLECTION:
   Always collect full name and order/subscription number before acting.
   Collect only what is needed — no unnecessary questions.

TONE: professional, empathetic, efficient. Max 2 questions per turn.

Always end your response with:
{"action": "<action>", "details": "<one sentence>"}

Valid actions: refund_or_reship | exchange_product | refund_duplicate |
               ship_correct_item | cancellation_confirmed | collect_data | escalate_human
"""

print('agent v2 loaded — targeted improvements')
print(f'prompt length: {len(AGENT_V2)} chars  (+{len(AGENT_V2)-len(AGENT_V1)} chars vs v1)')

## Step 11: Method B — Same Personas, Improved Prompt

We run the exact same personas against the improved prompt. The delta in resolution rate is the measurable value of the prompt changes — this is what shipping v2 instead of v1 is actually worth.

In [ ]:
results_v2 = run_eval(AGENT_V2, PERSONAS, label='Method B | Agent v2 (improved)')

gt_pass_v2 = sum(r['gt_pass'] for r in results_v2)
resolved_v2 = sum(r['resolved'] for r in results_v2)
avg_quality_v2 = sum(r['quality'] for r in results_v2) / len(results_v2)
avg_turns_v2 = sum(r['turns'] for r in results_v2) / len(results_v2)

print(f'\nImproved summary:')
print(f'  GT action match : {gt_pass_v2}/{len(results_v2)}')
print(f'  Resolved (judge): {resolved_v2}/{len(results_v2)}')
print(f'  Avg quality     : {avg_quality_v2:.1f}/5')
print(f'  Avg turns       : {avg_turns_v2:.1f}')

## Step 12: Compare v1 vs v2 — The Value of Each Prompt Change

In [ ]:
print('\n' + '='*68)
print('Prompt Improvement Results — v1 vs v2')
print('='*68)

rows = [
    ['GT accuracy (Method A)',   f'{gt_acc_v1:.0%}',            '(re-run v2 below if needed)'],
    ['GT action match (B)',      f'{gt_pass_v1}/{len(results_v1)}',  f'{gt_pass_v2}/{len(results_v2)}'],
    ['Resolution rate (judge)',  f'{resolved_v1}/{len(results_v1)}', f'{resolved_v2}/{len(results_v2)}'],
    ['Avg quality score',        f'{avg_quality_v1:.1f}/5',     f'{avg_quality_v2:.1f}/5'],
    ['Avg turns to resolve',     f'{avg_turns_v1:.1f}',         f'{avg_turns_v2:.1f}'],
]
print(tabulate(rows, headers=['Metric', 'v1 (baseline)', 'v2 (improved)'], tablefmt='rounded_outline'))

print('\nPer-persona delta:')
v1_by_id = {r['id']: r for r in results_v1}
v2_by_id = {r['id']: r for r in results_v2}
delta_rows = []
for pid in [p['id'] for p in PERSONAS]:
    r1 = v1_by_id.get(pid, {})
    r2 = v2_by_id.get(pid, {})
    q1 = r1.get('quality', 0)
    q2 = r2.get('quality', 0)
    res1 = 'yes' if r1.get('resolved') else 'no'
    res2 = 'yes' if r2.get('resolved') else 'no'
    delta = q2 - q1
    delta_str = f'+{delta}' if delta > 0 else str(delta)
    delta_rows.append([pid, r1.get('type', '')[:28], res1, res2, f'{q1}', f'{q2}', delta_str])
print(tabulate(delta_rows,
    headers=['ID', 'Persona type', 'Resolved v1', 'Resolved v2', 'Quality v1', 'Quality v2', 'Delta'],
    tablefmt='rounded_outline'))

# visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'Prompt Improvement Loop — v1 vs v2\n{MODEL} | TechStore Customer Service Agent',
             fontsize=12, fontweight='bold')

# chart 1: resolution rate
ax = axes[0]
categories = ['GT\nAction Match', 'Resolved\n(LLM Judge)']
v1_vals = [gt_pass_v1 / len(results_v1) * 100, resolved_v1 / len(results_v1) * 100]
v2_vals = [gt_pass_v2 / len(results_v2) * 100, resolved_v2 / len(results_v2) * 100]
x, w = range(len(categories)), 0.3
b1 = ax.bar([i - w/2 for i in x], v1_vals, w, label='v1 baseline', color='#E74C3C', alpha=0.85)
b2 = ax.bar([i + w/2 for i in x], v2_vals, w, label='v2 improved', color='#27AE60', alpha=0.85)
ax.set_xticks(list(x)); ax.set_xticklabels(categories, fontsize=9)
ax.set_ylim(0, 115); ax.set_ylabel('Pass rate (%)')
ax.set_title('Resolution Rate\nv1 vs v2')
ax.legend(fontsize=8)
for bars, vals in [(b1, v1_vals), (b2, v2_vals)]:
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                f'{v:.0f}%', ha='center', fontsize=9, fontweight='bold')
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# chart 2: quality scores per persona
ax = axes[1]
persona_ids = [p['id'] for p in PERSONAS]
q1_vals = [v1_by_id.get(pid, {}).get('quality', 0) for pid in persona_ids]
q2_vals = [v2_by_id.get(pid, {}).get('quality', 0) for pid in persona_ids]
xi = range(len(persona_ids))
ax.bar([i - w/2 for i in xi], q1_vals, w, label='v1 baseline', color='#E74C3C', alpha=0.85)
ax.bar([i + w/2 for i in xi], q2_vals, w, label='v2 improved', color='#27AE60', alpha=0.85)
ax.set_xticks(list(xi)); ax.set_xticklabels(persona_ids, fontsize=8)
ax.set_ylim(0, 6); ax.set_ylabel('Quality score (1-5)')
ax.set_title('Quality Score per Persona\nv1 vs v2')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

# chart 3: avg turns
ax = axes[2]
turns_v1 = [v1_by_id.get(pid, {}).get('turns', 0) for pid in persona_ids]
turns_v2 = [v2_by_id.get(pid, {}).get('turns', 0) for pid in persona_ids]
ax.bar([i - w/2 for i in xi], turns_v1, w, label='v1 baseline', color='#E74C3C', alpha=0.85)
ax.bar([i + w/2 for i in xi], turns_v2, w, label='v2 improved', color='#27AE60', alpha=0.85)
ax.set_xticks(list(xi)); ax.set_xticklabels(persona_ids, fontsize=8)
ax.set_ylabel('Turns to resolution')
ax.set_title('Turns to Resolution\n(fewer = more efficient)')
ax.legend(fontsize=8)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('prompt_improvement.png', dpi=150, bbox_inches='tight')
plt.show()
print('chart saved: prompt_improvement.png')

## Step 13: Final Verdict

The prompt improvement loop produces a measurable, defensible result: each change is traced to a specific failure, tested against the same adversarial personas, and quantified by an independent LLM judge. This is what differentiates systematic prompt engineering from iterating by intuition.

In [ ]:
resolved_delta = resolved_v2 - resolved_v1
quality_delta = avg_quality_v2 - avg_quality_v1
turns_delta = avg_turns_v2 - avg_turns_v1

print('=' * 66)
print('Final Verdict — Prompt Improvement Loop')
print('=' * 66)
print(f"""
Agent v1 (baseline)
  Resolution rate : {resolved_v1}/{len(results_v1)} personas fully resolved
  Avg quality     : {avg_quality_v1:.1f}/5
  Avg turns       : {avg_turns_v1:.1f}
  Key gaps        : no urgency protocol, no defect diagnosis step,
                    warranty/return conflation, no prior contact handling,
                    no friction minimization for cancellations

Agent v2 (improved)
  Resolution rate : {resolved_v2}/{len(results_v2)} personas fully resolved  ({resolved_delta:+d})
  Avg quality     : {avg_quality_v2:.1f}/5  ({quality_delta:+.1f})
  Avg turns       : {avg_turns_v2:.1f}  ({turns_delta:+.1f})
  Changes shipped : 5 targeted additions, each mapped to a diagnosed failure
""")

print('What each change was worth:')
change_map = [
    ('[A] B2B urgency protocol',       'P-001', 'eliminates unauthorized promises under pressure'),
    ('[B] Defect diagnosis step',       'P-002', 'catches symptom-described defects before wrong action'),
    ('[C] Warranty vs return policy',   'P-003', 'prevents 30-day window from blocking warranty claims'),
    ('[D] Prior contact handling',      'P-004', 'stops agent from asking repeat questions'),
    ('[E] Cancellation friction rules', 'P-005', 'prevents dropout from over-questioning'),
]
for change, pid, impact in change_map:
    r1 = v1_by_id.get(pid, {})
    r2 = v2_by_id.get(pid, {})
    res_change = ('no->yes' if not r1.get('resolved') and r2.get('resolved')
                  else 'yes->yes' if r1.get('resolved') and r2.get('resolved')
                  else 'no->no')
    q_delta = r2.get('quality', 0) - r1.get('quality', 0)
    print(f'  {change}')
    print(f'    persona {pid}: resolved={res_change}, quality {r1.get("quality",0)}->{r2.get("quality",0)} ({q_delta:+d})')
    print(f'    impact: {impact}')
    print()

print('Next steps for production:')
print('  1. Run pass^k: repeat each persona 5 times, measure consistency across runs')
print('  2. Add 10-20 more personas covering additional failure modes')
print('  3. Integrate into CI/CD: block deploy if resolution rate drops below threshold')
print('  4. Track quality score trend across prompt versions as a rolling baseline')
print('  5. Use failing persona transcripts as few-shot examples in v3')
print()
print('References:')
print('  tau-bench         : https://arxiv.org/abs/2406.12045')
print('  tau2-bench        : https://arxiv.org/abs/2506.07982')
print('  Zendesk CX 2024   : https://www.zendesk.com/cx-trends/')
print('  Intercom benchmark: https://www.intercom.com/resources/customer-support-benchmark')